# 实验五：真实数据跨菌株比对 (Real Data Cross-Strain Benchmark)

**数据源**  
- Query: DNBSEQ 测序的 *E. coli* JCM 5491 (DRR544745)，测序错误率 ~0.1%  
- Reference: *E. coli* K-12 MG1655 (ecoli.fa)  
- JCM 5491 与 K-12 之间存在约 2-4% 基因组序列差异（SNPs + Indels）

**子项 A: Mapping Rate** — 各方法在真实菌株变异下的有效比对率  
**子项 B: Recall w.r.t Union Set** — 相对于所有方法并集的召回率

In [ ]:
%matplotlib inline
import os
import sys
import time
import json
import csv
import gzip
import tempfile
import subprocess
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

import edlib
import matplotlib.pyplot as plt
import numpy as np

_nb_dir = Path.cwd()
if (_nb_dir / "v3_benchmark.ipynb").exists():
    METHODS_DIR = _nb_dir
elif (_nb_dir / "methods" / "v3_benchmark.ipynb").exists():
    METHODS_DIR = _nb_dir / "methods"
else:
    METHODS_DIR = _nb_dir
PROJECT_DIR = METHODS_DIR.parent
NAVIGAMER_CPP = PROJECT_DIR / "navigamer_cpp"
OUTPUT_DIR = METHODS_DIR / "v3_benchmark_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("METHODS_DIR:", METHODS_DIR)
print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# ── Method runners ──
def run_navigamer(ref_fa, query_fq, out_tsv, params):
    binary = NAVIGAMER_CPP / "navigamer"
    if not binary.exists():
        return {"error": f"navigamer not found at {binary}"}
    window = params.get("window_size", 200)
    stride = params.get("stride", 1)
    tol = params.get("tolerance", 5)
    r_sw = params.get("r_sw", 5)    # SW 层半径，默认 R_SW=5
    r_mw = params.get("r_mw", 15)   # MW 层半径，默认 R_MW=15
    r_lw = params.get("r_lw", 30)   # LW 层半径，默认 R_LW=30
    cmd = [str(binary), "benchmark", "--ref", ref_fa, "--reads", query_fq,
           "--tolerance", str(tol), "--window", str(window), "--stride", str(stride),
           "--r-sw", str(r_sw), "--r-mw", str(r_mw), "--r-lw", str(r_lw),
           "--out", out_tsv]
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd=str(NAVIGAMER_CPP))
    elapsed = time.time() - t0
    if proc.returncode != 0:
        return {"error": proc.stderr[:500], "runtime": elapsed}
    return parse_navigamer_output(out_tsv, elapsed)

def parse_navigamer_output(tsv_path, runtime=0.0):
    hits = defaultdict(list)
    total_leaf_verify = 0
    if not os.path.exists(tsv_path):
        return {"hits": hits, "candidates_count": 0, "runtime": runtime}
    with open(tsv_path) as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            rid = row.get("query_id") or row.get("read_id", "")
            if not rid:
                continue
            try:
                total_leaf_verify += int(row.get("leaf_verify_count", 0))
            except (ValueError, TypeError):
                pass
            ref_start = row.get("reference_start", "")
            if ref_start.isdigit() and row.get("hit_id"):
                hits[rid].append({"ref_start": int(ref_start), "ref_pos": int(ref_start),
                                "edit_dist": int(row.get("edit_distance") or 0), "score": int(row.get("score") or 0)})
    return {"hits": hits, "candidates_count": sum(len(v) for v in hits.values()), "runtime": runtime}

def run_strobemer(ref_fa, query_fq, out_tsv, params):
    strobe_dir = METHODS_DIR / "ksahlin" / "strobemers"
    cmd = [sys.executable, str(strobe_dir / "StrobeMap"), "--queries", query_fq, "--references", ref_fa,
           "--outfolder", str(Path(out_tsv).parent), "--prefix", Path(out_tsv).stem, "--k", str(params.get("k", 15)),
           "--strobe_w_min_offset", str(params.get("w_min", 20)), "--strobe_w_max_offset", str(params.get("w_max", 70)),
           "--w", str(params.get("w", 1)), "--n", str(params.get("order", 2)), "--rev_comp"]
    env = os.environ.copy()
    env["PYTHONPATH"] = str(strobe_dir) + os.pathsep + env.get("PYTHONPATH", "")
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True, env=env, cwd=str(strobe_dir))
    elapsed = time.time() - t0
    if proc.returncode != 0:
        return {"error": proc.stderr[:500], "hits": defaultdict(list), "candidates_count": 0, "runtime": elapsed}
    raw = Path(out_tsv).parent / (Path(out_tsv).stem + ".tsv")
    hits = parse_strobemer_output(str(raw))
    return {"hits": hits, "candidates_count": sum(len(v) for v in hits.values()), "runtime": elapsed}

def parse_strobemer_output(raw_path):
    results = defaultdict(list)
    if not os.path.exists(raw_path):
        return results
    current_query = None
    with open(raw_path) as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith(">"):
                parts = line[2:].strip().split()
                current_query = parts[0] if parts else None
            else:
                fields = line.strip().split()
                if len(fields) >= 4 and current_query:
                    ref_pos = int(fields[1])
                    query_pos = int(fields[2])
                    inferred_start = ref_pos - query_pos
                    results[current_query].append({
                        "ref_pos": inferred_start,
                        "ref_start": inferred_start
                    })
    return results

def run_spaced_seed(ref_fa, query_fq, out_tsv, params):
    ss_dir = METHODS_DIR / "spaced_seed"
    workers = params.get("workers", max(1, os.cpu_count() or 1))
    cmd = [sys.executable, str(ss_dir / "ales_fasta_fastq_align_parallel.py"), "--fasta", ref_fa, "--fastq", query_fq,
           "--output", out_tsv, "--weight", str(params.get("weight", 11)), "--homology-length", str(params.get("homology_length", 64)),
           "--similarity", str(params.get("similarity", 0.8)), "--k-seeds", str(params.get("k_seeds", 1)), "--no-revcomp",
           "--workers", "96"]
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ss_dir))
    elapsed = time.time() - t0
    if proc.returncode != 0:
        return {"error": proc.stderr[:500], "hits": defaultdict(list), "candidates_count": 0, "runtime": elapsed}
    hits = parse_spaced_seed_output(out_tsv)
    return {"hits": hits, "candidates_count": sum(len(v) for v in hits.values()), "runtime": elapsed}

def parse_spaced_seed_output(tsv_path):
    results = defaultdict(list)
    if not os.path.exists(tsv_path):
        return results
    with open(tsv_path) as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            rid = row.get("read_id", "")
            if rid and rid != "*":
                ref_start = int(row.get("reference_start", 0))
                results[rid].append({"ref_start": ref_start})
    return results

def run_tensor_sketch(ref_fa, query_fq, out_tsv, params):
    binary = METHODS_DIR / "tensor-sketch-alignment" / "metagraph" / "mgsketch_index"
    if not binary.exists():
        return {"error": "mgsketch_index not found", "hits": defaultdict(list), "candidates_count": 0, "runtime": 0}
    cmd = [str(binary), "--ref_fasta", ref_fa, "--query_fastq", query_fq, "--output_tsv", out_tsv,
           "--window_size", str(params.get("window_size", 150)), "--stride", str(params.get("stride", 1)),
           "--hnsw_ef_search", str(params.get("hnsw_ef_search", 50)), "--top_k", str(params.get("top_k", 5))]
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    if proc.returncode != 0:
        return {"error": proc.stderr[:500], "hits": defaultdict(list), "candidates_count": 0, "runtime": elapsed}
    hits = parse_tensor_sketch_output(out_tsv)
    return {"hits": hits, "candidates_count": sum(len(v) for v in hits.values()), "runtime": elapsed}

def parse_tensor_sketch_output(tsv_path):
    results = defaultdict(list)
    if not os.path.exists(tsv_path):
        return results
    with open(tsv_path) as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            rid = row.get("read_id", "")
            if rid:
                results[rid].append({"ref_start": int(row.get("reference_start", 0))})
    return results

In [ ]:
import edlib
import gzip

# ── Exp5 配置 ──
EXP5_SAMPLED_FASTQ = str(PROJECT_DIR / "data" / "ecoli" / "fastq" / "DRR544745_sampled_10k.fastq")
EXP5_REF_FASTA     = str(PROJECT_DIR / "data" / "ecoli" / "fasta" / "ecoli.fa")
EXP5_EDIT_THRESHOLD = 15   # ~90% identity for 150bp reads
EXP5_VERIFY_FLANK   = 20   # extra bases around candidate for semi-global alignment

# 加载完整参考基因组
def load_full_reference(fasta_path):
    seq_parts = []
    with open(fasta_path) as f:
        for line in f:
            if line.startswith(">"):
                continue
            seq_parts.append(line.strip().upper())
    return "".join(seq_parts)

# 加载真实 reads (FASTQ)
def load_real_reads(fastq_path):
    reads = {}
    opener = gzip.open if fastq_path.endswith('.gz') else open
    mode = 'rt' if fastq_path.endswith('.gz') else 'r'
    with opener(fastq_path, mode) as f:
        while True:
            header = f.readline().strip()
            if not header:
                break
            seq = f.readline().strip().upper()
            f.readline()  # +
            f.readline()  # quality
            rid = header.split()[0].lstrip('@')
            reads[rid] = seq
    return reads

# edlib 验证函数
def verify_candidate_edlib(query_seq, ref_seq, ref_start, threshold=EXP5_EDIT_THRESHOLD, flank=EXP5_VERIFY_FLANK):
    """Semi-global alignment: query against ref window with flanking."""
    qlen = len(query_seq)
    start = max(0, ref_start - flank)
    end = min(len(ref_seq), ref_start + qlen + flank)
    ref_sub = ref_seq[start:end]
    if len(ref_sub) < qlen // 2:
        return False, 999
    result = edlib.align(query_seq, ref_sub, mode="HW", task="distance")
    ed = result["editDistance"]
    if ed == -1:
        return False, 999
    return ed <= threshold, ed

print(f"Exp5 配置:")
print(f"  Sampled FASTQ: {EXP5_SAMPLED_FASTQ}")
print(f"  Reference: {EXP5_REF_FASTA}")
print(f"  Edit threshold: {EXP5_EDIT_THRESHOLD}")

exp5_ref = load_full_reference(EXP5_REF_FASTA)
exp5_reads = load_real_reads(EXP5_SAMPLED_FASTQ)
print(f"  Reference length: {len(exp5_ref):,} bp")
print(f"  Reads loaded: {len(exp5_reads):,}")

In [ ]:
# ── Exp5: 运行四个方法 ──
# 敏感模式参数 (针对 150bp reads 调整)
READ_LEN = 150
EXP5_PARAMS = {
    # window_size=150 匹配 read 长度; stride=10 保证最近窗口偏移 <10bp (远小于 tolerance)
    # tolerance=15 应对 ~2-4% 菌株差异 (150bp * 4% ≈ 6 mutations, 加上 stride 偏移留余量)
    "navigamer":     {"tolerance": 15, "window_size": READ_LEN, "stride": 10, "r_sw": 15, "r_mw": 20, "r_lw": 30},
    # k=15 较低以提高灵敏度; w_min/w_max 适配 150bp (偏移范围覆盖 read 中段)
    "strobemer":     {"k": 15, "order": 2, "w_min": 16, "w_max": 50},
    # weight=11 降低以提高灵敏度; homology_length=100 适配 150bp read
    "spaced_seed":   {"weight": 11, "similarity": 0.8, "homology_length": 100, "k_seeds": 1},
    # window_size=150 匹配 read 长度; top_k=50 高灵敏度
    "tensor_sketch": {"hnsw_ef_search": 200, "top_k": 50, "window_size": READ_LEN, "tuple_size": 3},
}

exp5_raw_hits = {}

with tempfile.TemporaryDirectory() as tmpdir:
    # 写入 FASTQ 供各方法使用
    query_fq = EXP5_SAMPLED_FASTQ
    ref_fa = EXP5_REF_FASTA
    
    for method_name in ["navigamer", "strobemer", "spaced_seed", "tensor_sketch"]:
        print(f"\n{'='*60}")
        print(f"Running {method_name} ...")
        out_tsv = os.path.join(tmpdir, f"exp5_{method_name}.tsv")
        params = EXP5_PARAMS[method_name]
        
        t0 = time.time()
        if method_name == "navigamer":
            res = run_navigamer(ref_fa, query_fq, out_tsv, params)
        elif method_name == "strobemer":
            res = run_strobemer(ref_fa, query_fq, out_tsv, params)
        elif method_name == "spaced_seed":
            res = run_spaced_seed(ref_fa, query_fq, out_tsv, params)
        else:
            res = run_tensor_sketch(ref_fa, query_fq, out_tsv, params)
        elapsed = time.time() - t0
        
        if "error" in res:
            print(f"  ERROR: {res['error']}")
        
        hits = res.get("hits", {})
        n_reads_with_hits = len(hits)
        n_total_hits = sum(len(v) for v in hits.values())
        print(f"  Time: {elapsed:.1f}s")
        print(f"  Reads with raw hits: {n_reads_with_hits}")
        print(f"  Total raw candidate hits: {n_total_hits}")
        
        exp5_raw_hits[method_name] = hits

print("\n" + "="*60)
print("All methods completed.")

In [ ]:
# ── Exp5: edlib 验证 + 计算指标 ──
from concurrent.futures import ProcessPoolExecutor, as_completed

def _verify_worker(args):
    """Worker: verify all candidates for one read under one method."""
    rid, query_seq, hit_list, ref_seq, threshold, flank = args
    for hit in hit_list:
        ref_start = hit.get("ref_start", hit.get("ref_pos", -1))
        if ref_start < 0:
            continue
        ok, ed = verify_candidate_edlib(query_seq, ref_seq, ref_start, threshold, flank)
        if ok:
            return rid, True, ed
    return rid, False, 999

method_valid_reads = {}
method_best_ed = {}
union_set = set()
total_input = len(exp5_reads)

for method_name, hits in exp5_raw_hits.items():
    print(f"\nVerifying {method_name} ...")
    valid_reads = set()
    best_eds = {}
    
    tasks = []
    for rid, hit_list in hits.items():
        if rid not in exp5_reads:
            continue
        tasks.append((rid, exp5_reads[rid], hit_list, exp5_ref, EXP5_EDIT_THRESHOLD, EXP5_VERIFY_FLANK))
    
    with ProcessPoolExecutor(max_workers=min(96, os.cpu_count() or 4)) as pool:
        futures = {pool.submit(_verify_worker, t): t[0] for t in tasks}
        done_count = 0
        for future in as_completed(futures):
            rid, is_valid, ed = future.result()
            if is_valid:
                valid_reads.add(rid)
                best_eds[rid] = ed
            done_count += 1
            if done_count % 2000 == 0:
                print(f"  Verified {done_count}/{len(tasks)} reads ...")
    
    method_valid_reads[method_name] = valid_reads
    method_best_ed[method_name] = best_eds
    union_set |= valid_reads
    print(f"  {method_name}: {len(valid_reads)} verified reads out of {len(hits)} with raw hits")

print(f"\n{'='*60}")
print(f"Total input reads: {total_input}")
print(f"Union set size: {len(union_set)}")

# ── 计算 Mapping Rate 和 Recall w.r.t Union ──
exp5_results = {}
for method_name in ["navigamer", "strobemer", "spaced_seed", "tensor_sketch"]:
    valid = method_valid_reads.get(method_name, set())
    mapping_rate = len(valid) / total_input if total_input > 0 else 0
    recall_union = len(valid) / len(union_set) if len(union_set) > 0 else 0
    exp5_results[method_name] = {
        "verified_count": len(valid),
        "mapping_rate": mapping_rate,
        "recall_union": recall_union,
        "raw_hit_reads": len(exp5_raw_hits.get(method_name, {})),
    }
    print(f"\n{method_name}:")
    print(f"  Raw reads with hits: {exp5_results[method_name]['raw_hit_reads']}")
    print(f"  Verified reads:      {len(valid)}")
    print(f"  Mapping Rate:        {mapping_rate:.4f} ({mapping_rate*100:.2f}%)")
    print(f"  Recall (Union):      {recall_union:.4f} ({recall_union*100:.2f}%)")

# Unique hits 分析
nav_valid = method_valid_reads.get("navigamer", set())
seed_methods_valid = method_valid_reads.get("strobemer", set()) | method_valid_reads.get("spaced_seed", set())
nav_unique = nav_valid - seed_methods_valid
print(f"\n{'='*60}")
print(f"NavigaMer unique hits (not found by Strobemer or Spaced Seed): {len(nav_unique)}")
if len(nav_unique) > 0:
    print(f"  (These are reads 'rescued' by NavigaMer)")

# 保存结果
exp5_output = {
    "config": {
        "sampled_reads": total_input,
        "edit_threshold": EXP5_EDIT_THRESHOLD,
        "verify_flank": EXP5_VERIFY_FLANK,
        "params": {k: str(v) for k, v in EXP5_PARAMS.items()},
    },
    "union_set_size": len(union_set),
    "results": exp5_results,
    "navigamer_unique_vs_seed": len(nav_unique),
}
with open(OUTPUT_DIR / "exp5_real_data.json", "w") as f:
    json.dump(exp5_output, f, indent=2)
print(f"\nResults saved to {OUTPUT_DIR / 'exp5_real_data.json'}")

In [ ]:
# ── Exp5: 可视化 ──
_methods_order = ["navigamer", "strobemer", "spaced_seed", "tensor_sketch"]
_labels = ["NavigaMer", "Strobemer", "Spaced Seed", "Tensor Sketch"]
_colors = ["#4C72B0", "#55A868", "#DD8452", "#C44E52"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# ── 子图 A: Mapping Rate (柱状图) ──
ax = axes[0]
rates = [exp5_results[m]["mapping_rate"] * 100 for m in _methods_order]
bars = ax.bar(range(len(_methods_order)), rates, color=_colors, alpha=0.85, edgecolor="white", linewidth=1.2)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{rate:.1f}%", ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(range(len(_methods_order)))
ax.set_xticklabels(_labels, fontsize=9, rotation=10)
ax.set_ylabel("Mapping Rate (%)", fontsize=11)
ax.set_title("Exp5a: Mapping Rate\n(Real Cross-Strain Data)", fontsize=12, fontweight="bold")
ax.set_ylim(0, max(rates) * 1.15 if rates else 100)
ax.grid(True, alpha=0.2, axis="y")

# ── 子图 B: Recall w.r.t Union Set (柱状图) ──
ax = axes[1]
recalls = [exp5_results[m]["recall_union"] * 100 for m in _methods_order]
bars = ax.bar(range(len(_methods_order)), recalls, color=_colors, alpha=0.85, edgecolor="white", linewidth=1.2)
for bar, recall in zip(bars, recalls):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{recall:.1f}%", ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(range(len(_methods_order)))
ax.set_xticklabels(_labels, fontsize=9, rotation=10)
ax.set_ylabel("Recall w.r.t Union (%)", fontsize=11)
ax.set_title("Exp5b: Recall w.r.t Union Set\n(Higher = More Complete)", fontsize=12, fontweight="bold")
ax.set_ylim(0, 105)
ax.axhline(100, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)
ax.grid(True, alpha=0.2, axis="y")

# ── 子图 C: Verified vs Raw counts (分组柱状图) ──
ax = axes[2]
x = np.arange(len(_methods_order))
width = 0.35
raw_counts = [exp5_results[m]["raw_hit_reads"] for m in _methods_order]
verified_counts = [exp5_results[m]["verified_count"] for m in _methods_order]

bars1 = ax.bar(x - width/2, raw_counts, width, label="Raw Hits", color=[c + "80" for c in _colors],
               edgecolor=_colors, linewidth=1.2)
bars2 = ax.bar(x + width/2, verified_counts, width, label="Verified Hits", color=_colors, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(_labels, fontsize=9, rotation=10)
ax.set_ylabel("Number of Reads", fontsize=11)
ax.set_title("Exp5c: Raw vs Verified Hits\n(Precision Indicator)", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis="y")

plt.suptitle(f"Exp5: Real Data Cross-Strain Benchmark  (n={total_input:,} reads, ED≤{EXP5_EDIT_THRESHOLD})",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "exp5_real_data.png", bbox_inches='tight', dpi=150)
plt.show()
print(f"Figure saved to {OUTPUT_DIR / 'exp5_real_data.png'}")